# Module 2: Principal Component Analysis (PCA)

This module will talk about a method for reducing the dimensionality of data in applications; namely, principal component analysis (PCA).

### Overiew

Working with *big* data is difficult for a variety of reasons. Put simply, it is harder to analyze and interpret large amounts of data from both a practical perspective and a computational perspective. Thankfully, there are techniques to exploit the structure in high-dimensional data. For example, high-dimensional data oftentimes has features that are redundant. Moreover, some of these features can be explained by linear combinations of other features, and thus can be removed from the data. What this leads to is the ability to extract a low-dimensional substructure of the original high-dimensional data. PCA is a technique to do exactly this.

PCA is a well established technique that can help us indentify simple patterns, latent factors, and structure in high-dimensional data. We now derive this algorithm from first principles.

## Problem Formulation

Let $(\Omega, \mathcal{F}, \mathbb{P})$ be a probability space and let 

$$
    X_n: \Omega \to \mathbb{R}^K
$$

be independent and identically distributed random vectors with finite second moments for $1 \le n \le N$. To make our notational life easy, we assume without loss of generality that each random vector $X_n$ has mean zero (apply a translation by $-\mathbb{E}[X_1] = -\mathbb{E}[X_n]$ for $1 \le n \le N$). Let $\omega_0 \in \Omega$ and consider the observed dataset 

$$
    \mathcal{D} = (X_1(\omega_0), \ldots, X_N(\omega_0)) =: (\mathbf{x}_1, \ldots, \mathbf{x}_N),
$$

where we denote

$$
    \mathbf{x}_n = (x_{n,1}, \ldots, x_{n,K}).
$$

We may view our data as the matrix

$$
    \mathcal{X} = (x_{n,k})_{1\le n \le N, 1 \le k \le K}.
$$

Let $M \in \mathbb{N}$ with $0 < M < K$. We are interested in finding the "best" orthogonal projection of the data vectors $\mathbf{x}_n$ onto an $M$-dimensional subspace $U$ of $\mathbb{R}^K$. We call a subspace $U$ corresponding to such an optimal projection a **principal subspace**. 

For any subspace $U$ of $\mathbb{R}^K$ with dimension $M$, we know that we may find an ordered orthonormal basis $(\mathbf{b}_1, \ldots, \mathbf{b}_M)$ of $U$. Put

$$
    \mathbf{z}_n := (\langle \mathbf{x}_n, \mathbf{b}_1\rangle_{\mathbb{R}^K}, \ldots, \langle \mathbf{x}_n, \mathbf{b}_M\rangle_{\mathbb{R}^K}) \in \mathbb{R}^M, \quad 1\le n \le N.
$$

We call the $\mathbf{z}_n$ the *codes* of $\mathbf{x}_n$ with respect to the subspace $U$ and orthonormal basis $(\mathbf{b}_1, \ldots, \mathbf{b}_M)$. For the given subspace $U$, we know that 

$$
    \tilde{\mathbf{x}}_n := \sum^M_{m=1} z_{n,m}\mathbf{b}_m = \argmin_{\mathbf{x} \in U}||\mathbf{x}_n - \mathbf{x}||^2_{\mathbb{R}^K}.
$$

Thus our main problem becomes *find an optimal subspace* $U$ *of* $\mathbb{R}^K$ *with* $\operatorname{dim}(U) = M$ *such that the average reconstruction loss*

$$
    J_M(U):=\frac{1}{N}\sum^N_{n=1} || \mathbf{x}_n - \tilde{\mathbf{x}}_n||^2_{\mathbb{R}^K}
$$

*is as small as possible.*  Note here that $J_M(U)$ admits the same value for any choice of orthonormal basis of $U$ by the uniqueness of the orthogonal projection.
 
Now let us define what we mean by *reconstruction loss*.

### Encoders, Decoders, and the Reconstruction Loss

Let $U \subseteq \mathbb{R}^K$ be an $M$-dimensional subspace, and let $(\mathbf{b}_1,\ldots,\mathbf{b}_M)$ be an orthonormal basis of $U$. 


<p align="center">
  <img src="figures/encoderdecoder.png" width="700">
</p>


Consider the **encoding map**

$$
\operatorname{Enc}_U:\mathbb{R}^K \to \mathbb{R}^M,
\qquad
\operatorname{Enc}_U(\mathbf{x})
=
(\langle \mathbf{x},\mathbf{b}_1\rangle,\ldots,\langle \mathbf{x},\mathbf{b}_M\rangle).
$$

From the code $\mathbf{z}_n=(z_{n,1},\ldots,z_{n,M})$, we recover an approximation of $\mathbf{x}_n$ by defining the **decoding map**

$$
\operatorname{Dec}_U:\mathbb{R}^M \to U,
\qquad
\operatorname{Dec}_U(z_1,\ldots,z_M)
=
\sum_{m=1}^M z_m \mathbf{b}_m.
$$

Applying the decoder to the code of $\mathbf{x}_n$ yields

$$
\tilde{\mathbf{x}}_n
:=
\operatorname{Dec}_U(\mathbf{z}_n)
=
\sum_{m=1}^M z_{n,m}\mathbf{b}_m
=
\sum_{m=1}^M
\langle \mathbf{x}_n,\mathbf{b}_m\rangle \mathbf{b}_m
=
P_U(\mathbf{x}_n),
$$

the orthogonal projection of $\mathbf{x}_n$ onto $U$.

We call $\tilde{\mathbf{x}}_n$ the **reconstruction** of $\mathbf{x}_n$. The quantity
$$
\|\mathbf{x}_n-\tilde{\mathbf{x}}_n\|_{\mathbb{R}^K}^2
$$

is called the **reconstruction error** of $\mathbf{x}_n$ (with respect to $U$); i.e., the squared Euclidean distance between the original data vector and its projection onto the subspace $U$.The average reconstruction error over the dataset is therefore

$$
J_M(U)
=
\frac{1}{N}\sum_{n=1}^N
\|\mathbf{x}_n-\tilde{\mathbf{x}}_n\|_{\mathbb{R}^K}^2.
$$

Thus, we can intuitively see that PCA seeks an $M$-dimensional subspace $U$ for which the encoding-decoding process loses as little information as possible on average.

### Minimizing the Reconstruction Loss

Suppose $(\mathbf{b}_1, \ldots, \mathbf{b}_K)$ is an ordered orthonormal basis for $\mathbb{R}^K$. Denote by $\mathbf{v} \otimes \mathbf{v}$ the outer product of the two vectors, and define

$$
        S:= \frac{1}{N}\mathcal{X}^t\mathcal{X} = \frac{1}{N}\sum^N_{n=1} (\mathbf{x}_n \otimes \mathbf{x}_n).
$$

For $M \in \mathbb{N}$, $1 \le M < K$, let $U = \operatorname{span}(\mathbf{b}_1, \ldots, \mathbf{b}_M)$, and observe that

$$
    \mathbf{x}_n - \tilde{\mathbf{x}}_n = \sum_{j = M+1}^K \langle \mathbf{x}_n, \mathbf{b}_j \rangle_{\mathbb{R}^K}\mathbf{b}_j = \sum_{j=M+1}^K (\mathbf{b}_j \otimes \mathbf{b}_j)\mathbf{x}_n = P_{U^\perp}(\mathbf{x}_n).
$$

But then,

$$
    \begin{align*}

        J_M(U) = \frac{1}{N} \sum^N_{n=1} || \mathbf{x}_n - \tilde{\mathbf{x}}_n ||^2_{\mathbb{R}^K} &= \frac{1}{N}\sum^N_{n=1} \left|\left| \sum^K_{j = M+1} \langle \mathbf{x}_n, \mathbf{b}_j \rangle_{\mathbb{R}^K} \mathbf{b}_j \right|\right|^2_{\mathbb{R}^K}\\
        &= \frac{1}{N} \sum^N_{n=1} \sum^K_{j = M+1} |\langle \mathbf{x}_n, \mathbf{b}_j \rangle_{\mathbb{R}^K}|^2\\
        &= \frac{1}{N} \sum^N_{n=1} \sum^K_{j = M+1} \langle \mathbf{b}_j, (\mathbf{x}_n \otimes \mathbf{x}_n)\mathbf{b}_j\rangle_{\mathbb{R}^K}\\
        &=  \sum^K_{j = M+1} \left\langle \mathbf{b}_j, \left(\frac{1}{N}\sum^N_{n=1} \mathbf{x}_n \otimes \mathbf{x}_n\right)\mathbf{b}_j\right\rangle_{\mathbb{R}^K}\\
        &=  \sum^K_{j = M+1} \langle \mathbf{b}_j, S\mathbf{b}_j \rangle_{\mathbb{R}^K}.
    \end{align*}
$$

 It is clear that $S \in \mathbb{R}^{K \times K}$ is positive semidefinite and symmetric. By the spectral theorem, we know there exists an orthonormal basis $\mathbf{q}_1, \ldots, \mathbf{q}_K$ of $\mathbb{R}^K$ that are the eigenvectors of $S$, with corresponding eigenvalues $\lambda_1 \ge  \cdots \ge \lambda_K \ge 0$. We make a couple of observations:

 1. We have that for $1 \le m \le K$,
 
 $$
 \mathbf{b}_m = \sum^K_{k=1} \langle \mathbf{b}_m, \mathbf{q}_k \rangle_{\mathbb{R}^K}\mathbf{q}_k \implies S\mathbf{b}_m = S\sum^K_{k=1} \langle \mathbf{b}_m, \mathbf{q}_k \rangle_{\mathbb{R}^K}\mathbf{q}_k = \sum^K_{k=1} \langle \mathbf{b}_m, \mathbf{q}_k \rangle_{\mathbb{R}^K} S\mathbf{q}_k = \sum^K_{k=1} \lambda_k \langle \mathbf{b}_m, \mathbf{q}_k \rangle_{\mathbb{R}^K} \mathbf{q}_k;
 $$
 
 2. Moreover, for $1 \le m \le K$,

 $$
    \langle S\mathbf{b}_m, \mathbf{b}_m\rangle_{\mathbb{R}^K} = \left\langle \sum^K_{k=1} \lambda_k \langle \mathbf{b}_m, \mathbf{q}_k \rangle_{\mathbb{R}^K} \mathbf{q}_k, \mathbf{b}_m \right\rangle_{\mathbb{R}^K} = \sum^K_{k=1} \lambda_k |\langle \mathbf{b}_m, \mathbf{q}_k \rangle_{\mathbb{R}^K}|^2.
 $$

 With these observations in mind, we establish that

 $$
    J_M(U) = \sum^K_{j = M+1}\sum^K_{k=1} \lambda_k|\langle \mathbf{b}_j, \mathbf{q}_k \rangle_{\mathbb{R}^K}|^2 = \sum^{K}_{k=1}\lambda_k\sum^K_{j = M+1} |\langle \mathbf{b}_j, \mathbf{q}_k \rangle_{\mathbb{R}^K}|^2 = \sum^K_{k=1}\lambda_k\alpha_k,
 $$

 where 

 $$
    \alpha_k:= \sum^K_{j = M+1}|\langle \mathbf{b}_j, \mathbf{q}_k \rangle_{\mathbb{R}^K}|^2 = || P_{U^\perp}(\mathbf{q}_k) ||^2_{\mathbb{R}^K} \le 1,
 $$

 as $P_{U^\perp}$ is a contraction. Thus we have that 

 $$
        0 \le \alpha_k \le 1, \quad 1 \le k \le K.
 $$

 Moreover, by Parseval's identity, we have

 $$
        \sum^K_{k=1} \alpha_k = \sum^K_{j = M+1} \sum^K_{k=1} |\langle \mathbf{b}_j, \mathbf{q}_k \rangle_{\mathbb{R}^K}|^2 = \sum^K_{j = M+1} || \mathbf{b}_j ||^2_{\mathbb{R}^K} = K- M.
 $$

So for every choice of $U$ with dimension $M$, we have that

$$
    0 \le \alpha_k \le 1, \quad 1 \le k \le K \quad \text{ and } \quad \sum^K_{k=1} \alpha_k = K-M.
$$

Our objective now is to minimize 

$$ 
    J_M(U) = \sum^K_{k=1}\lambda_k \alpha_k.
$$

Observe that by the ordering of the eigenvalues

$$
    J_M(U) - \sum^K_{j = M+1} \lambda_j = \sum^K_{k=1}\lambda_k\alpha_k - \sum^K_{j =M+1}\lambda_j = \sum^M_{k =1}\lambda_k\alpha_k - \sum^K_{j = M+1}\lambda_j(1-\alpha_j) \ge \lambda_M\sum^M_{k=1}\alpha_k - \lambda_M\sum^K_{j = M+1}(1-\alpha_j) = \lambda_M\left(\sum^K_{k=1}\alpha_k - (K-M)\right) = 0.
$$

In particular, we have that for any subspace $U$ of $\mathbb{R}^K$ with $\operatorname{dim}(U) = M$,

$$
    J_M(U) \ge \sum^K_{j= M+1}\lambda_j.
$$

Note that this lower bound is independent of the choice of $U$. The next proposition shows that $J_M$ attains this lower bound.

**Proposition.** *Let* $M \in \mathbb{N}$ *such that* $1 \le M < K$*, and let* $(\mathbf{q}_1, \ldots, \mathbf{q}_K)$ *be the orthonormal basis of eigenvectors as we have above. Then for* $U^*:= \operatorname{span}(\mathbf{q}_1, \ldots, \mathbf{q}_M)$, *we have* 

$$
       J_M(U^*) = \sum^K_{j = M+1} \lambda_{j}; 
$$

*that is,* $J_M$ *attains its minimum at* $U^*$. *In particular,* $U^*$ *is a principal subspace.*

*Proof.* Simply observe that

$$
    J_M(U^*) = \sum^K_{j = M+1} \langle S\mathbf{q}_j, \mathbf{q}_j\rangle_{\mathbb{R}^K} = \sum^K_{j = M+1} \lambda_j || \mathbf{q}_j ||^2_{\mathbb{R}^K} = \sum^K_{j = M+1}\lambda_j,
$$

as desired. $\blacksquare$

# Probabilistic Principal Component Analysis (PPCA)

We now develop a latent-variable probabilistic perspective of PCA with the tools that we have estabslished.

### The Probabilistic Approach